# Multi-layer perceptron: intuition
Two perceptrons, one perceptron that combines them, and small MLPs on non-linear data.

In [ ]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# the sigmoid squeezes any number into the range 0 to 1
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

## Combining two probabilities

In [ ]:
p1, p2 = 0.7, 0.8
# plain sum, then sigmoid
print(round(sigmoid(p1 + p2), 3))            # 0.818
# weighted sum plus bias, then sigmoid
z = 10 * p1 + 5 * p2 + 3
print(z, sigmoid(z))                          # 14.0, 0.99999917

## The combination that bends the boundary
Perceptron 1: $\sigma(3x_1 + 3x_2)$, perceptron 2: $\sigma(3x_1 - 3x_2)$, combiner: $\sigma(8p_1 + 8p_2 - 12)$.

In [ ]:
def p1(x1, x2): return sigmoid(3 * x1 + 3 * x2)
def p2(x1, x2): return sigmoid(3 * x1 - 3 * x2)
def combined(x1, x2): return sigmoid(8 * p1(x1, x2) + 8 * p2(x1, x2) - 12)

# (0, 0) lies on both lines: each perceptron says 0.5, the combination says 0.018
for pt in [(1, 0), (0, 0)]:
    print(pt, round(p1(*pt), 3), round(p2(*pt), 3), round(combined(*pt), 3))

In [ ]:
# probability maps: drag the slider to move the combiner's bias and watch the boundary change
xs = np.linspace(-3, 3, 151)
X1, X2 = np.meshgrid(xs, xs)
fig = go.Figure()
biases = [-15, -12, -9, -6, -3]
for b in biases:
    Z = sigmoid(8 * p1(X1, X2) + 8 * p2(X1, X2) + b)
    fig.add_trace(go.Contour(x=xs, y=xs, z=Z, zmin=0, zmax=1, visible=(b == -12),
                             colorscale=[[0, "#F6C9C9"], [0.5, "#FFFFFF"], [1, "#CBE5C5"]],
                             contours=dict(start=0.1, end=0.9, size=0.1)))
steps = [dict(method="update", label=str(b), args=[{"visible": [c == b for c in biases]}]) for b in biases]
fig.update_layout(sliders=[dict(active=1, steps=steps, currentvalue=dict(prefix="combiner bias "))],
                  width=600, height=600, template="simple_white")
fig.show()

## The 2-2-1 network of Figure 2

In [ ]:
def mlp_221(cgpa, iq):
    h1 = sigmoid(2 * cgpa + 3 * iq + 6)     # perceptron 1
    h2 = sigmoid(5 * cgpa + 4 * iq + 3)     # perceptron 2
    return sigmoid(10 * h1 + 5 * h2 + 3)   # perceptron 3 combines them

# with these made-up positive weights every student gets almost 1:
# the numbers only show the structure, they are not trained
print(mlp_221(7.5, 110))

## XOR with two hidden nodes
Four noisy clusters around the four XOR inputs (centred at ±1.5). Two sigmoid hidden nodes, one sigmoid output node, plain gradient descent on the log loss: learning rate 1, 5,000 steps, small random starting weights (standard deviation 0.01). Code in `images/playground.py`.

In [ ]:
import sys
sys.path.insert(0, "images")
from playground import datasets, fit_xor, fit_all, TwoNodeNet

X, y, net = fit_xor(seed=0)
print("accuracy:", net.score(X, y))
# each hidden node draws the line w1*x1 + w2*x2 + b = 0
for j in range(2):
    w1, w2 = net.W1[:, j]
    print(f"hidden node {j + 1}: {w1:.2f} x1 + {w2:.2f} x2 + {net.b1[j]:.2f} = 0")
print("output node: weights", net.w2.round(2), "bias", round(net.b2, 2))

In [ ]:
# what each node says about one point from each cluster (corner order: (-,-), (+,+), (-,+), (+,-))
corners = np.array([[-1.5, -1.5], [1.5, 1.5], [-1.5, 1.5], [1.5, -1.5]])
print(np.c_[corners, net.hidden(corners), net.predict_proba(corners)[:, 1]].round(3))

In [ ]:
# the three probability maps: hidden node 1, hidden node 2, output
g = np.linspace(-3, 3, 121)
G1, G2 = np.meshgrid(g, g)
grid = np.c_[G1.ravel(), G2.ravel()]
maps = [net.hidden(grid)[:, 0], net.hidden(grid)[:, 1], net.predict_proba(grid)[:, 1]]
fig = make_subplots(1, 3, subplot_titles=["hidden node 1", "hidden node 2", "output"])
for k, Z in enumerate(maps, start=1):
    fig.add_trace(go.Contour(x=g, y=g, z=Z.reshape(G1.shape), showscale=False,
                             colorscale=[[0, "#F6C9C9"], [1, "#CBE5C5"]]), 1, k)
    fig.add_trace(go.Scatter(x=X[:, 0], y=X[:, 1], mode="markers", showlegend=False,
                             marker=dict(color=np.where(y == 1, "#54A24B", "#E45756"), size=4)), 1, k)
fig.update_layout(width=1000, height=380, template="simple_white")
fig.show()

## Over many random starts
The same training from 100 different random starting weights.

In [ ]:
acc = [TwoNodeNet(seed=s).fit(X, y).score(X, y) for s in range(100)]
print("starts reaching 100%:", sum(a == 1 for a in acc), "of 100; lowest accuracy", min(acc))

## TensorFlow Playground demos with scikit-learn
The same data and models as the circles and spirals figure.

In [ ]:
runs = fit_all()
for title, Xr, yr, clf in runs:
    print(f"{title:35s} accuracy {clf.score(Xr, yr):.2f}")

In [ ]:
# decision boundaries, one panel per model
fig = make_subplots(1, 3, subplot_titles=[t for t, *_ in runs])
g = np.linspace(-4, 4, 150)
G1, G2 = np.meshgrid(g, g)
for k, (title, Xr, yr, clf) in enumerate(runs, start=1):
    Z = clf.predict_proba(np.c_[G1.ravel(), G2.ravel()])[:, 1].reshape(G1.shape)
    fig.add_trace(go.Contour(x=g, y=g, z=Z, showscale=False,
                             colorscale=[[0, "#F6C9C9"], [1, "#CBE5C5"]]), 1, k)
    fig.add_trace(go.Scatter(x=Xr[:, 0], y=Xr[:, 1], mode="markers", showlegend=False,
                             marker=dict(color=np.where(yr == 1, "#54A24B", "#E45756"), size=4)), 1, k)
fig.update_layout(width=1100, height=400, template="simple_white")
fig.show()

## Try it
Change `hidden_layer_sizes` or `activation` and refit.

In [ ]:
import warnings
warnings.filterwarnings("ignore")
from sklearn.neural_network import MLPClassifier
Xc, yc = datasets()["circles"]
for h in [(1,), (2,), (3,), (4,), (8,)]:
    clf = MLPClassifier(hidden_layer_sizes=h, activation="logistic", solver="lbfgs",
                        max_iter=5000, random_state=0).fit(Xc, yc)
    print(h, round(clf.score(Xc, yc), 3))

## Checking the solvers over 5 random starts

In [ ]:
data = datasets()
for name, h, act in [("circles", (4,), "logistic"), ("spiral", (4, 4, 4, 4), "relu"),
                     ("spiral", (4, 4, 4, 4), "logistic")]:
    Xd, yd = data[name]
    for solver in ["lbfgs", "adam"]:
        acc = [round(MLPClassifier(h, activation=act, solver=solver, max_iter=5000, random_state=s).fit(Xd, yd).score(Xd, yd), 2)
               for s in range(5)]
        print(name, act, solver, acc)